# BÜYÜK VERİ FİNAL PROJESİ

## PySpark ve LDA Kullanılarak Haber Verileri Üzerinde Konu Modelleme

### İsim:
Egemen Gündüz

### Numara:
25281907

### Proje Konusu:
Haber Verileri ile Konu Modelleme

### Kullanılan Teknolojiler:
Apache Spark, PySpark, Spark MLlib, Hadoop HDFS, Docker, Jupyter Notebook

# Özet

# İçindekiler

1. Giriş
2. Problem Tanımı
3. Literatür
4. Veri Seti
5. Yöntem
6. Büyük Veri İşleme Süreci
7. Deneysel Sonuçlar
8. Tartışma
9. Sonuç
10. Kaynakça

# 1. Giriş

# 2. Problem Tanımı

# 3. Literatür

# 4. Veri Seti

Bu projede Kaggle platformunda yer alan “News Category Dataset” veri seti kullanılmıştır.

**Veri seti bağlantısı**:

https://www.kaggle.com/datasets/rmisra/news-category-dataset

Veri seti yukarıdaki linkten zip formatında indirilir ve extract edilir. "News_Category_Dataset_v3.json" isimli dosya notebook ile aynı dizine getirilir. **Bu işlem ön kontrol için yapılıyor, daha sonra dağıtık depolama ve işleme yapılacaktır.**

Veri seti; haber başlıkları, kısa açıklamalar, kategori bilgileri ve tarih verilerinden oluşmaktadır. Veri seti büyük ölçekli metin verisi içerdiğinden dolayı Apache Spark ile dağıtık veri işleme süreçleri için uygun yapıdadır.

In [1]:
import pandas as pd

In [2]:
df_p = pd.read_json(
    "News_Category_Dataset_v3.json",
    lines=True
)

In [3]:
df_p.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


# 5. Yöntem

## 5.1 Kullanılan Yöntemler
## 5.2 TF-IDF Yaklaşımı
## 5.3 LDA Algoritması

# 6. Büyük Veri İşleme Süreci

## 6.1 SparkSession Oluşturulması
Bu aşamada PySpark kullanılarak SparkSession oluşturulmuştur. SparkSession, Spark uygulamasının başlangıç noktasıdır ve DataFrame işlemleri, SQL işlemleri ve Spark MLlib süreçleri için temel arayüz sağlar.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, concat_ws, trim, length

In [5]:
import subprocess

namenode_ip = subprocess.check_output(
    "docker inspect -f '{{range.NetworkSettings.Networks}}{{.IPAddress}}{{end}}' namenode",
    shell=True,
    text=True
).strip()

In [6]:
spark = SparkSession.builder \
    .appName("HaberAnalizi") \
    .master("spark://localhost:7077") \
    .config("spark.driver.host", "172.18.0.1") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.port", "4045") \
    .config("spark.blockManager.port", "4046") \
    .config("spark.hadoop.fs.defaultFS", f"hdfs://{namenode_ip}:9000") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "false") \
    .config("spark.executor.memory", "512m") \
    .config("spark.cores.max", "2") \
    .config("spark.executor.extraJavaOptions", "-Dhadoop.security.logger=ERROR,RFAS") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark Master:", sc.master)
print("Default Parallelism:", sc.defaultParallelism)

26/05/21 11:44:26 WARN Utils: Your hostname, egemen-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/05/21 11:44:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 11:44:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Master: spark://localhost:7077
Default Parallelism: 2


In [7]:
data_path = f"hdfs://{namenode_ip}:9000/bigdata/news/input/News_Category_Dataset_v3.json"

df = spark.read.json(data_path)

print("Toplam kayıt:", df.count())
print("Partition sayısı:", df.rdd.getNumPartitions())

[Stage 1:>                                                          (0 + 2) / 2]

Toplam kayıt: 209527
Partition sayısı: 2


## 6.2 Veri Setinin PySpark ile Okunması

News Category Dataset JSON formatında PySpark DataFrame olarak okunmuştur. Bu işlem pandas yerine Spark kullanılarak gerçekleştirilmiştir.

In [8]:
df.limit(5).toPandas()

,authors,category,date,headline,link,short_description
0,"Carla K. Johnson, AP",U.S. NEWS,2022-09-23,Over 4 Million Americans Roll Up Sleeves For O...,https://www.huffpost.com/entry/covid-boosters-...,Health experts said it is too early to predict...
1,Mary Papenfuss,U.S. NEWS,2022-09-23,"American Airlines Flyer Charged, Banned For Li...",https://www.huffpost.com/entry/american-airlin...,He was subdued by passengers and crew when he ...
2,Elyse Wanshel,COMEDY,2022-09-23,23 Of The Funniest Tweets About Cats And Dogs ...,https://www.huffpost.com/entry/funniest-tweets...,"""Until you have a dog you don't understand wha..."
3,Caroline Bologna,PARENTING,2022-09-23,The Funniest Tweets From Parents This Week (Se...,https://www.huffpost.com/entry/funniest-parent...,"""Accidentally put grown-up toothpaste on my to..."
4,Nina Golgowski,U.S. NEWS,2022-09-22,Woman Who Called Cops On Black Bird-Watcher Lo...,https://www.huffpost.com/entry/amy-cooper-lose...,Amy Cooper accused investment firm Franklin Te...


In [9]:
from pyspark.sql.functions import col

category_counts = df.groupBy("category") \
    .count() \
    .orderBy(col("count").desc())

category_counts.limit(20).toPandas()

,category,count
0,POLITICS,35602
1,WELLNESS,17945
2,ENTERTAINMENT,17362
3,TRAVEL,9900
4,STYLE & BEAUTY,9814
5,PARENTING,8791
6,HEALTHY LIVING,6694
7,QUEER VOICES,6347
8,FOOD & DRINK,6340
9,BUSINESS,5992


## 6.3 Veri Ön İşleme


In [10]:
df_selected = df.select(
    "category",
    "headline",
    "short_description",
    "date"
)

df_selected.limit(5).toPandas()

,category,headline,short_description,date
0,U.S. NEWS,Over 4 Million Americans Roll Up Sleeves For O...,Health experts said it is too early to predict...,2022-09-23
1,U.S. NEWS,"American Airlines Flyer Charged, Banned For Li...",He was subdued by passengers and crew when he ...,2022-09-23
2,COMEDY,23 Of The Funniest Tweets About Cats And Dogs ...,"""Until you have a dog you don't understand wha...",2022-09-23
3,PARENTING,The Funniest Tweets From Parents This Week (Se...,"""Accidentally put grown-up toothpaste on my to...",2022-09-23
4,U.S. NEWS,Woman Who Called Cops On Black Bird-Watcher Lo...,Amy Cooper accused investment firm Franklin Te...,2022-09-22


In [11]:
df_selected = df_selected.withColumn(
    "text",
    concat_ws(" ", "headline", "short_description")
)

In [12]:
df_selected = df_selected.withColumn(
    "text",
    lower("text")
)

In [13]:
df_selected = df_selected.withColumn(
    "text",
    regexp_replace("text", r"http\S+|[^a-zA-Z\s]", "")
)

In [14]:
df_selected.limit(5).toPandas()

,category,headline,short_description,date,text
0,U.S. NEWS,Over 4 Million Americans Roll Up Sleeves For O...,Health experts said it is too early to predict...,2022-09-23,over million americans roll up sleeves for om...
1,U.S. NEWS,"American Airlines Flyer Charged, Banned For Li...",He was subdued by passengers and crew when he ...,2022-09-23,american airlines flyer charged banned for lif...
2,COMEDY,23 Of The Funniest Tweets About Cats And Dogs ...,"""Until you have a dog you don't understand wha...",2022-09-23,of the funniest tweets about cats and dogs th...
3,PARENTING,The Funniest Tweets From Parents This Week (Se...,"""Accidentally put grown-up toothpaste on my to...",2022-09-23,the funniest tweets from parents this week sep...
4,U.S. NEWS,Woman Who Called Cops On Black Bird-Watcher Lo...,Amy Cooper accused investment firm Franklin Te...,2022-09-22,woman who called cops on black birdwatcher los...


## 6.4 Tokenization

In [15]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="text",
    outputCol="tokens"
)

df_tokenized = tokenizer.transform(df_selected)

In [16]:
df_tokenized.select(
    "text",
    "tokens"
).limit(5).toPandas()

,text,tokens
0,over million americans roll up sleeves for om...,"[over, , million, americans, roll, up, sleeves..."
1,american airlines flyer charged banned for lif...,"[american, airlines, flyer, charged, banned, f..."
2,of the funniest tweets about cats and dogs th...,"[, of, the, funniest, tweets, about, cats, and..."
3,the funniest tweets from parents this week sep...,"[the, funniest, tweets, from, parents, this, w..."
4,woman who called cops on black birdwatcher los...,"[woman, who, called, cops, on, black, birdwatc..."


In [17]:
from pyspark.sql.functions import size

df_tokenized.select(
    "text",
    size("tokens").alias("token_count")
).limit(10).toPandas()

,text,token_count
0,over million americans roll up sleeves for om...,40
1,american airlines flyer charged banned for lif...,41
2,of the funniest tweets about cats and dogs th...,25
3,the funniest tweets from parents this week sep...,34
4,woman who called cops on black birdwatcher los...,36
5,cleaner was dead in belk bathroom for days be...,39
6,reporter gets adorable surprise from her boyfr...,31
7,puerto ricans desperate for water after hurric...,28
8,how a new documentary captures the complexity ...,35
9,biden at un to call russian war an affront to ...,38


## 6.5 Stopwords Temizleme

In [31]:
import nltk
from nltk.corpus import stopwords
from pyspark.ml.feature import StopWordsRemover

nltk.download("stopwords")

spark_stopwords = StopWordsRemover.loadDefaultStopWords("english")
nltk_stopwords = stopwords.words("english")

custom_stopwords = [
    "said", "new", "one", "time", "day", "week",
    "best", "photos", "photo", "video", "people",
    "make", "get", "like", "may"
]

all_stopwords = list(set(
    spark_stopwords +
    nltk_stopwords +
    custom_stopwords
))

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens",
    stopWords=all_stopwords
)

df_filtered = remover.transform(df_tokenized)

df_filtered.select(
    "tokens",
    "filtered_tokens"
).limit(5).toPandas()

[nltk_data] Downloading package stopwords to /home/egemen/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,tokens,filtered_tokens
0,"[over, , million, americans, roll, up, sleeves...","[, million, americans, roll, sleeves, omicront..."
1,"[american, airlines, flyer, charged, banned, f...","[american, airlines, flyer, charged, banned, l..."
2,"[, of, the, funniest, tweets, about, cats, and...","[, funniest, tweets, cats, dogs, sept, , dog, ..."
3,"[the, funniest, tweets, from, parents, this, w...","[funniest, tweets, parents, sept, , accidental..."
4,"[woman, who, called, cops, on, black, birdwatc...","[woman, called, cops, black, birdwatcher, lose..."


In [32]:
from pyspark.sql.functions import size

df_filtered.select(
    size("tokens").alias("token_count_before"),
    size("filtered_tokens").alias("token_count_after")
).limit(10).toPandas()

,token_count_before,token_count_after
0,40,22
1,41,22
2,25,11
3,34,19
4,36,24
5,39,24
6,31,18
7,28,19
8,35,19
9,38,22


## 6.6 TF-IDF Özellik Çıkarımı

In [33]:
from pyspark.ml.feature import CountVectorizer, IDF

cv = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    vocabSize=5000,
    minDF=5
)

cv_model = cv.fit(df_filtered)

df_cv = cv_model.transform(df_filtered)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

idf_model = idf.fit(df_cv)

df_tfidf = idf_model.transform(df_cv)

df_tfidf.select(
    "category",
    "filtered_tokens",
    "features"
).limit(5).toPandas()

,category,filtered_tokens,features
0,U.S. NEWS,"[, million, americans, roll, sleeves, omicront...","(2.3853370601527875, 2.738101899032303, 0.0, 0..."
1,U.S. NEWS,"[american, airlines, flyer, charged, banned, l...","(0.0, 2.738101899032303, 0.0, 3.27862773427532..."
2,COMEDY,"[, funniest, tweets, cats, dogs, sept, , dog, ...","(2.3853370601527875, 0.0, 0.0, 0.0, 0.0, 3.331..."
3,PARENTING,"[funniest, tweets, parents, sept, , accidental...","(1.1926685300763937, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,U.S. NEWS,"[woman, called, cops, black, birdwatcher, lose...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


## 6.7 LDA Modelinin Eğitilmesi

In [42]:
from pyspark.ml.clustering import LDA

lda = LDA(
    k=10,
    maxIter=10,
    featuresCol="features",
    seed=42
)

lda_model = lda.fit(df_tfidf)

In [43]:
ll = lda_model.logLikelihood(df_tfidf)
lp = lda_model.logPerplexity(df_tfidf)

print("Log Likelihood:", ll)
print("Log Perplexity:", lp)

[Stage 104:============================>                            (1 + 1) / 2]

Log Likelihood: -115473279.4789859
Log Perplexity: 7.872384419623195


# 7. Deneysel Sonuçlar

## 7.1 Kategori Dağılımı

In [44]:
from pyspark.sql.functions import col

category_counts = (
    df.groupBy("category")
    .count()
    .orderBy(col("count").desc())
)

category_counts.limit(20).toPandas()

,category,count
0,POLITICS,35602
1,WELLNESS,17945
2,ENTERTAINMENT,17362
3,TRAVEL,9900
4,STYLE & BEAUTY,9814
5,PARENTING,8791
6,HEALTHY LIVING,6694
7,QUEER VOICES,6347
8,FOOD & DRINK,6340
9,BUSINESS,5992


## 7.2 Konular ve Anahtar Kelimeler

In [45]:
topics = lda_model.describeTopics(10)
topics.toPandas()

,topic,termIndices,termWeights
0,0,"[0, 43, 85, 2, 195, 694, 96, 720, 12, 9]","[0.0069814807886564405, 0.004692670667659664, ..."
1,1,"[2, 0, 12, 271, 25, 387, 1, 9, 32, 374]","[0.008761601508962588, 0.005889131361684909, 0..."
2,2,"[0, 22, 60, 1, 124, 230, 377, 382, 457, 75]","[0.007669299517544208, 0.004775013876812688, 0..."
3,3,"[0, 151, 427, 2, 16, 249, 628, 4, 112, 56]","[0.005230145416721214, 0.004156683566141862, 0..."
4,4,"[0, 213, 1, 2, 469, 26, 126, 588, 1064, 9]","[0.00736581018109019, 0.003571427275840302, 0...."
5,5,"[0, 115, 321, 235, 54, 611, 430, 186, 300, 870]","[0.0063920136859543725, 0.005946110905822395, ..."
6,6,"[0, 3, 45, 20, 140, 6, 180, 51, 38, 68]","[0.008753122987384582, 0.006382220042518971, 0..."
7,7,"[0, 5, 163, 111, 400, 10, 17, 33, 3, 536]","[0.006557429059023852, 0.00499275917292927, 0...."
8,8,"[110, 133, 92, 87, 0, 101, 119, 433, 454, 10]","[0.007118315422458983, 0.007000831577151993, 0..."
9,9,"[0, 214, 310, 15, 8, 26, 138, 134, 11, 21]","[0.011105590280502266, 0.00430616093936228, 0...."


In [46]:
vocab = cv_model.vocabulary

def get_topic_words(term_indices):
    return [vocab[i] for i in term_indices]

topics_pd = topics.toPandas()

topics_pd["words"] = (
    topics_pd["termIndices"]
    .apply(get_topic_words)
)

topics_pd[["topic","words"]]

,topic,words
0,0,"[, show, york, trump, report, jimmy, city, lon..."
1,1,"[trump, , donald, north, president, gun, us, s..."
2,2,"[, health, state, us, states, law, tax, federa..."
3,3,"[, court, supreme, trump, women, students, sch..."
4,4,"[, rights, us, trump, paul, kids, men, general..."
5,5,"[, star, hair, film, white, awards, stories, c..."
6,6,"[, life, never, take, baby, years, sleep, litt..."
7,7,"[, dont, hillary, clinton, sanders, want, need..."
8,8,"[check, facebook, style, twitter, , huffpost, ..."
9,9,"[, holiday, weight, year, know, kids, summer, ..."


## 7.3 Konu Dağılımları

In [47]:
df_topics = lda_model.transform(df_tfidf)

df_topics.select(
    "headline",
    "topicDistribution"
).limit(10).toPandas()

,headline,topicDistribution
0,Over 4 Million Americans Roll Up Sleeves For O...,"[0.2202031366678879, 0.0011439390950748409, 0...."
1,"American Airlines Flyer Charged, Banned For Li...","[0.11256387586476756, 0.4096298603518818, 0.00..."
2,23 Of The Funniest Tweets About Cats And Dogs ...,"[0.0019593120749849206, 0.001958306944334561, ..."
3,The Funniest Tweets From Parents This Week (Se...,"[0.001343955329116576, 0.11318362898879358, 0...."
4,Woman Who Called Cops On Black Bird-Watcher Lo...,"[0.46195971933680663, 0.26527594977153834, 0.0..."
5,Cleaner Was Dead In Belk Bathroom For 4 Days B...,"[0.18235163800174317, 0.8106322725667038, 0.00..."
6,Reporter Gets Adorable Surprise From Her Boyfr...,"[0.4397999054748488, 0.19722891791831854, 0.00..."
7,Puerto Ricans Desperate For Water After Hurric...,"[0.0013975766492660823, 0.0013969301788663519,..."
8,How A New Documentary Captures The Complexity ...,"[0.0017552106712317036, 0.0017543301083783928,..."
9,Biden At UN To Call Russian War An Affront To ...,"[0.001060709212060601, 0.7176395181081674, 0.0..."


In [48]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import expr

df_topics = df_topics.withColumn(
    "topic_array",
    vector_to_array("topicDistribution")
)

df_topics = df_topics.withColumn(
    "dominant_topic",
    expr(
        "array_position(topic_array,array_max(topic_array))-1"
    )
)

In [49]:
topic_words = {
    row["topic"]: ", ".join(row["words"][:3])
    for _, row in topics_pd.iterrows()
}

df_topics_pd = (
    df_topics
    .select(
        "headline",
        "dominant_topic"
    )
    .limit(20)
    .toPandas()
)

df_topics_pd["dominant_topic_text"] = (
    df_topics_pd["dominant_topic"]
    .map(topic_words)
)

df_topics_pd[
    ["headline","dominant_topic_text"]
]

,headline,dominant_topic_text
0,Over 4 Million Americans Roll Up Sleeves For O...,", health, state"
1,"American Airlines Flyer Charged, Banned For Li...","trump, , donald"
2,23 Of The Funniest Tweets About Cats And Dogs ...,", holiday, weight"
3,The Funniest Tweets From Parents This Week (Se...,", holiday, weight"
4,Woman Who Called Cops On Black Bird-Watcher Lo...,", show, york"
5,Cleaner Was Dead In Belk Bathroom For 4 Days B...,"trump, , donald"
6,Reporter Gets Adorable Surprise From Her Boyfr...,", show, york"
7,Puerto Ricans Desperate For Water After Hurric...,", health, state"
8,How A New Documentary Captures The Complexity ...,", star, hair"
9,Biden At UN To Call Russian War An Affront To ...,"trump, , donald"


# 8. Tartışma

# 9. Sonuç

# 10. Kaynakça